# CNN Assignment — Reading the Road: Traffic Sign Recognition

**Level:** Intermediate · **Suggested time:** 1–2 weeks · **Frameworks:** TensorFlow/Keras *or* PyTorch

## The scenario
You have joined the perception team at a self-driving car startup. The car's camera captures the road, and your job is to build a CNN that reads a cropped photo of a single traffic sign and says *which of 43 signs it is*. The data (GTSRB) is genuinely messy — variable size, lighting, blur, and class imbalance — and that is exactly where the interesting decisions live.

> **How this notebook works:** each step gives you a little starter code, a set of **Hints**, and a **TODO** checklist. You write the rest. Pick one framework and stick with it. Set a seed, use a GPU (Colab: *Runtime → Change runtime type → GPU*).


### Setup
**Hints**
- Import only what your chosen framework needs; delete the other block.
- Fix every random seed you can (`numpy`, and your framework's) so results are reproducible.

**TODO**
- [ ] Import your libraries
- [ ] Set a global `SEED`
- [ ] Detect and print whether a GPU is available


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

# import your framework and set its seed; detect GPU  (PyTorch chosen)
import os, random
import torch, torchvision
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__)
print('GPU available:', torch.cuda.is_available(), '| using device:', device)

---
## Part A — Explore the data  *(10%)*

**Hints**
- Load with `torchvision.datasets.GTSRB(root, split='train', download=True)` **or** `tensorflow_datasets.load('gtsrb')`, or a Kaggle folder/CSV.
- To build a label list fast in torchvision, the `(path, label)` pairs live in `dataset._samples`.
- Imbalance matters later (Part D): note which classes are rare *now*.
- Don't load every image just to measure sizes — sample a few hundred.

**TODO**
- [ ] A1: load train + test, print how many images in each and how many classes
- [ ] A2: show a 5×5 grid of random training images **with their labels**
- [ ] A3: plot a histogram of class frequencies; name the 3 rarest classes
- [ ] A4: report min / median / max image width and height


In [ ]:
# A1
# load the datasets, then set NUM_CLASSES and print the counts
from collections import Counter

CLASS_NAMES = [
    'Speed limit 20', 'Speed limit 30', 'Speed limit 50', 'Speed limit 60', 'Speed limit 70',
    'Speed limit 80', 'End of speed limit 80', 'Speed limit 100', 'Speed limit 120', 'No passing',
    'No passing (>3.5t)', 'Right-of-way at next intersection', 'Priority road', 'Yield', 'Stop',
    'No vehicles', 'Vehicles >3.5t prohibited', 'No entry', 'General caution', 'Dangerous curve left',
    'Dangerous curve right', 'Double curve', 'Bumpy road', 'Slippery road', 'Road narrows (right)',
    'Road work', 'Traffic signals', 'Pedestrians', 'Children crossing', 'Bicycles crossing',
    'Beware ice/snow', 'Wild animals crossing', 'End of all limits', 'Turn right ahead', 'Turn left ahead',
    'Ahead only', 'Go straight or right', 'Go straight or left', 'Keep right', 'Keep left',
    'Roundabout mandatory', 'End of no passing', 'End of no passing (>3.5t)']

DATA_ROOT = './data'
train_raw = torchvision.datasets.GTSRB(root=DATA_ROOT, split='train', download=True)
test_raw  = torchvision.datasets.GTSRB(root=DATA_ROOT, split='test',  download=True)

# (path, label) pairs live in dataset._samples
train_labels = np.array([lbl for _, lbl in train_raw._samples])
test_labels  = np.array([lbl for _, lbl in test_raw._samples])

NUM_CLASSES = 43
assert len(CLASS_NAMES) == NUM_CLASSES and len(set(train_labels)) == NUM_CLASSES
print(f'Train images: {len(train_raw)}')
print(f'Test images : {len(test_raw)}')
print(f'Classes     : {NUM_CLASSES}')

In [ ]:
# A2 — grid of 25 random images with labels
# Hint: fig, axes = plt.subplots(5, 5, ...); loop, pick a random index, imshow, set_title(label)
rng = np.random.default_rng(SEED)
idxs = rng.choice(len(train_raw), size=25, replace=False)

fig, axes = plt.subplots(5, 5, figsize=(11, 11))
for ax, i in zip(axes.ravel(), idxs):
    img, lbl = train_raw[int(i)]
    ax.imshow(img)
    ax.set_title(f'{lbl}: {CLASS_NAMES[lbl][:20]}', fontsize=7)
    ax.axis('off')
plt.suptitle('25 random training images', y=0.92)
plt.tight_layout()
plt.show()

In [ ]:
# A3 — class-frequency histogram
# Hint: from collections import Counter; count your label list; plt.bar(...)
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

plt.figure(figsize=(13, 4))
plt.bar(range(NUM_CLASSES), class_counts, color='steelblue')
plt.xlabel('Class id'); plt.ylabel('Number of training images')
plt.title('Class frequencies in the GTSRB training set')
plt.xticks(range(0, NUM_CLASSES, 2))
plt.show()

RARE_CLASSES = [int(c) for c in np.argsort(class_counts)[:3]]
print('3 rarest classes:')
for c in RARE_CLASSES:
    print(f'  class {c:2d} ({CLASS_NAMES[c]}): {class_counts[c]} images')
print(f'\nMost common class has {class_counts.max()} images -> imbalance ratio ≈ {class_counts.max() / class_counts.min():.1f}x')

In [ ]:
# A4 — image size distribution (sample a few hundred images)
# Hint: a PIL image exposes .size as (width, height)
from PIL import Image

sample_idx = np.random.default_rng(SEED).choice(len(train_raw), size=500, replace=False)
sizes = np.array([Image.open(train_raw._samples[int(i)][0]).size for i in sample_idx])   # (width, height)
widths, heights = sizes[:, 0], sizes[:, 1]

print(f'Width : min={widths.min()}, median={int(np.median(widths))}, max={widths.max()}')
print(f'Height: min={heights.min()}, median={int(np.median(heights))}, max={heights.max()}')

plt.figure(figsize=(6, 4))
plt.scatter(widths, heights, s=8, alpha=0.5)
plt.xlabel('Width (px)'); plt.ylabel('Height (px)'); plt.title('Image sizes (500 sampled)')
plt.show()

---
## Part B — Preprocess  *(10%)*

**Hints**
- Resize everything to one square size (32×32 or 48×48 are sensible). **Justify your choice in the report.**
- Scale pixels into [0,1] (torchvision `ToTensor()` already does this) or standardize per channel.
- Make the val split **stratified by class** so rare signs don't vanish from validation. A plain random split can drop a rare class entirely.
- Shuffle the *training* loader only; keep val/test in fixed order.

**TODO**
- [ ] B1/B2: define a resize+normalize transform (and a separate augmenting one later, in Part D)
- [ ] B3: build a stratified 80/20 train/val split
- [ ] B4: create train / val / test loaders (or `tf.data` pipelines) with batching


In [ ]:
# B1 + B2 — transforms / preprocessing
IMG_SIZE = 32
BATCH_SIZE = 64
# 32x32 is enough to distinguish sign shapes/symbols (median raw size is ~40-50 px),
# keeps training fast, and gives 8x8 feature maps after two 2x2 pools.

# Fixed constants (not computed from test data) -> no leakage
MEAN, STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),                    # scales pixels to [0, 1]
    transforms.Normalize(MEAN, STD),          # -> [-1, 1]
])

# re-load the datasets with the transform
train_full = torchvision.datasets.GTSRB(root=DATA_ROOT, split='train', transform=eval_tf)
test_ds    = torchvision.datasets.GTSRB(root=DATA_ROOT, split='test',  transform=eval_tf)
print('One sample tensor:', train_full[0][0].shape, '| range:',
      float(train_full[0][0].min()), 'to', float(train_full[0][0].max()))

In [ ]:
# B3 — stratified 80/20 train/val split
# Hint: group training indices by class, shuffle each group, take 20% of EACH class for val.
#       from collections import defaultdict; then torch Subset / sklearn train_test_split(stratify=...)
from sklearn.model_selection import train_test_split

all_idx = np.arange(len(train_full))
train_idx, val_idx = train_test_split(
    all_idx, test_size=0.2, stratify=train_labels, random_state=SEED)

train_ds = Subset(train_full, train_idx)
val_ds   = Subset(train_full, val_idx)

val_counts = np.bincount(train_labels[val_idx], minlength=NUM_CLASSES)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print(f'Every class present in val? {bool((val_counts > 0).all())} (smallest class has {val_counts.min()} val images)')
# Note: GTSRB images come in "tracks" of ~30 frames of the same physical sign, so a random split puts
# near-duplicates in train and val -> val accuracy is slightly optimistic. Test accuracy is the honest number.

In [ ]:
# B4 — data loaders
# Hint (PyTorch): DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True) for train; shuffle=False otherwise
NUM_WORKERS = 2
PIN = (device.type == 'cuda')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN)

xb, yb = next(iter(train_loader))
print('Batch:', xb.shape, yb.shape)

---
## Part C — Baseline CNN  *(25%)*

**Hints**
- Keep the baseline small: 2 blocks of Conv→ReLU→MaxPool, then Flatten→Dense→output.
- Output layer has **43 units**. In PyTorch leave them as raw logits (`CrossEntropyLoss` applies softmax internally); in Keras use `softmax` + `sparse_categorical_crossentropy`.
- Work out the flattened size: with IMG_SIZE=32 and two 2×2 pools you get 8×8 feature maps.
- Record **both** train and val metrics every epoch — you need them for the curves.

**TODO**
- [ ] C1: define the baseline CNN from scratch (no pretrained weights)
- [ ] C2: write/compile a training loop that logs train + val loss & accuracy per epoch
- [ ] C3: plot the learning curves
- [ ] C4: report **test accuracy** and interpret the curves (2–3 sentences)


In [ ]:
# C1 — baseline architecture (fill in the layers)
class BaselineCNN(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES, img_size=IMG_SIZE):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 32x32 -> 16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16x16 -> 8x8
        )
        flat = 64 * (img_size // 4) ** 2           # 64 * 8 * 8 = 4096
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 128), nn.ReLU(),
            nn.Linear(128, n_classes),             # raw logits (CrossEntropyLoss applies softmax)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

    def cam_layer(self):                           # layer used later for Grad-CAM (last ReLU)
        return self.features[-2]

print(BaselineCNN())
print('Parameters:', sum(p.numel() for p in BaselineCNN().parameters()))

In [ ]:
# C2 — train the baseline, logging train & val metrics each epoch
EPOCHS = 15
# Hint (PyTorch): one function run_epoch(loader, train=bool) that returns (loss, acc);
#   remember model.train()/model.eval(), optimizer.zero_grad(), loss.backward(), optimizer.step()

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over `loader`. Trains if an optimizer is given, otherwise evaluates. Returns (loss, accuracy)."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if training:
                optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

def fit(model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3, criterion=None,
        weight_decay=0.0, cosine=False, restore_best=False, verbose=True):
    """Train and log train/val loss+accuracy every epoch. Validation always uses plain (unweighted) CE."""
    model.to(device)
    train_criterion = nn.CrossEntropyLoss() if criterion is None else criterion
    eval_criterion = nn.CrossEntropyLoss()
    if weight_decay > 0:
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if cosine else None

    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_state = -1.0, None
    for ep in range(1, epochs + 1):
        tl, ta = run_epoch(model, train_loader, train_criterion, opt)
        vl, va = run_epoch(model, val_loader, eval_criterion)
        if sched: sched.step()
        for k, v in zip(hist, (tl, ta, vl, va)): hist[k].append(v)
        if va > best_acc:
            best_acc = va
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        if verbose:
            print(f'Epoch {ep:2d}/{epochs} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f}')
    if restore_best and best_state is not None:
        model.load_state_dict(best_state)
    return hist

set_seed()
baseline = BaselineCNN().to(device)
base_hist = fit(baseline, train_loader, val_loader, epochs=EPOCHS, lr=1e-3)

In [ ]:
# C3 — plot loss and accuracy curves (train vs val)
def plot_history(hist, title=''):
    ep = range(1, len(hist['train_loss']) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, hist['train_loss'], label='train'); ax[0].plot(ep, hist['val_loss'], label='val')
    ax[0].set_title(f'{title} — loss'); ax[0].set_xlabel('Epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].plot(ep, hist['train_acc'], label='train'); ax[1].plot(ep, hist['val_acc'], label='val')
    ax[1].set_title(f'{title} — accuracy'); ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

plot_history(base_hist, 'Baseline CNN')

In [ ]:
# C4 — evaluate on the test set and print accuracy
base_test_loss, base_test_acc = run_epoch(baseline, test_loader, nn.CrossEntropyLoss())
print(f'Baseline test loss: {base_test_loss:.4f} | test accuracy: {base_test_acc:.4f}')

**C4 interpretation (write here):** _Overfitting, underfitting, or healthy? What in the curves tells you?_

---
## Part D — Make it better  *(30%)*

Apply **at least three** techniques and measure **each** against the baseline.

**Hints**
- Change **one thing at a time** or you won't know what helped.
- Augmentation ideas: rotation, small shifts, zoom, brightness. **Do NOT flip horizontally** — it can turn a left-turn sign into a right-turn sign.
- Overfitting (val ≪ train)? → augmentation, dropout, batchnorm. Underfitting? → bigger/deeper net, train longer.
- Imbalance: pass class weights to the loss, or oversample rare classes.
- Transfer learning (MobileNetV2 / ResNet) is a strong comparison point.

**TODO**
- [ ] Pick ≥3 techniques and run them, logging val accuracy for each
- [ ] Fill in the comparison table below (**this table is the heart of the assignment**)
- [ ] Keep your best model for Part E


In [ ]:
# D — experiments (one change at a time)
# Hint (PyTorch augmentation, no flip):
#   transforms.Compose([Resize, RandomRotation(15), ColorJitter(0.2,0.2),
#                        RandomAffine(0, translate=(0.1,0.1)), ToTensor()])
import pandas as pd

# ---- A deeper CNN with BatchNorm + Dropout (used in Exp 2 and the final model) ----
class ImprovedCNN(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES, img_size=IMG_SIZE):
        super().__init__()
        def block(cin, cout, p):
            return [nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(),
                    nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(),
                    nn.MaxPool2d(2), nn.Dropout2d(p)]
        self.features = nn.Sequential(*block(3, 32, 0.1), *block(32, 64, 0.2), *block(64, 128, 0.3))
        flat = 128 * (img_size // 8) ** 2           # three pools -> 4x4
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(flat, 256), nn.ReLU(),
                                        nn.Dropout(0.4), nn.Linear(256, n_classes))
    def forward(self, x):
        return self.classifier(self.features(x))
    def cam_layer(self):                           # last ReLU before the final pool (8x8 map)
        return self.features[-3]

# ---- Augmentation pipeline (NO horizontal flip: left/right signs would swap meaning) ----
aug_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.RandomAffine(0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
train_aug_full = torchvision.datasets.GTSRB(root=DATA_ROOT, split='train', transform=aug_tf)
train_aug_loader = DataLoader(Subset(train_aug_full, train_idx), batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN)   # same train indices, augmented

# ---- Class weights (inverse frequency) for the imbalance ----
train_counts = np.bincount(train_labels[train_idx], minlength=NUM_CLASSES)
class_w = torch.tensor(train_counts.sum() / (NUM_CLASSES * train_counts), dtype=torch.float32, device=device)
weighted_ce = nn.CrossEntropyLoss(weight=class_w)

base_val = max(base_hist['val_acc'])
results = [('Baseline (Part C)', '—', base_val)]
hists = {'Baseline': base_hist}

print('\n===== Exp 1: data augmentation (baseline CNN) =====')
set_seed(); m1 = BaselineCNN()
h1 = fit(m1, train_aug_loader, val_loader, verbose=False); hists['Exp 1'] = h1
results.append(('Exp 1', 'Data augmentation (rotation, shift, zoom, brightness/contrast; no flip)', max(h1['val_acc'])))
print('best val acc:', round(max(h1['val_acc']), 4))

print('\n===== Exp 2: BatchNorm + Dropout + deeper net =====')
set_seed(); m2 = ImprovedCNN()
h2 = fit(m2, train_loader, val_loader, verbose=False); hists['Exp 2'] = h2
results.append(('Exp 2', 'Deeper CNN with BatchNorm + Dropout', max(h2['val_acc'])))
print('best val acc:', round(max(h2['val_acc']), 4))

print('\n===== Exp 3: class-weighted loss (baseline CNN) =====')
set_seed(); m3 = BaselineCNN()
h3 = fit(m3, train_loader, val_loader, criterion=weighted_ce, verbose=False); hists['Exp 3'] = h3
results.append(('Exp 3', 'Inverse-frequency class weights in the loss', max(h3['val_acc'])))
print('best val acc:', round(max(h3['val_acc']), 4))

# ---- Final model: combine ONLY the techniques that beat the baseline, then train longer ----
keep_aug, keep_bn, keep_w = max(h1['val_acc']) > base_val, max(h2['val_acc']) > base_val, max(h3['val_acc']) > base_val
print(f'\nKept for final model -> augmentation: {keep_aug} | BatchNorm/Dropout: {keep_bn} | class weights: {keep_w}')
FINAL_EPOCHS = 25
set_seed()
final_model = ImprovedCNN() if keep_bn else BaselineCNN()
final_hist = fit(final_model, train_aug_loader if keep_aug else train_loader, val_loader,
                 epochs=FINAL_EPOCHS, lr=2e-3, criterion=weighted_ce if keep_w else None,
                 weight_decay=1e-4, cosine=True, restore_best=True, verbose=True)
final_val = max(final_hist['val_acc'])
final_desc = ' + '.join([n for n, k in [('augmentation', keep_aug), ('BN/Dropout', keep_bn), ('class weights', keep_w)] if k] or ['—']) \
             + f', AdamW + cosine LR, {FINAL_EPOCHS} epochs'
results.append(('Final model', final_desc, final_val))
plot_history(final_hist, 'Final model')

# ---- Comparison table ----
table = pd.DataFrame(results, columns=['Experiment', 'Change made', 'Val accuracy'])
table['Δ vs baseline'] = (table['Val accuracy'] - base_val).map(lambda d: f'{d:+.4f}')
table['Val accuracy'] = table['Val accuracy'].round(4)
table.loc[0, 'Δ vs baseline'] = '—'
print('\n', table.to_string(index=False))

final_test_loss, final_test_acc = run_epoch(final_model, test_loader, nn.CrossEntropyLoss())
print(f'\nFinal model test accuracy: {final_test_acc:.4f}   (baseline: {base_test_acc:.4f})')
torch.save(final_model.state_dict(), 'best_traffic_sign_cnn.pt')

### Part D results table (fill in)

| Experiment | Change made | Val accuracy | Δ vs baseline |
|---|---|---|---|
| Baseline (Part C) | — | ___ | — |
| Exp 1 | ___ | ___ | ___ |
| Exp 2 | ___ | ___ | ___ |
| Exp 3 | ___ | ___ | ___ |
| **Final model** | ___ | ___ | ___ |


---
## Part E — Evaluate & analyze errors  *(15%)*

**Hints**
- First collect `(y_true, y_pred)` over the whole test set, then everything else is easy.
- `sklearn.metrics.confusion_matrix` and `classification_report` do the heavy lifting.
- "Most-confused pairs" = the largest **off-diagonal** entries of the confusion matrix.
- Look at the actual misclassified images — are the confused signs genuinely similar, or just rare/low-res?

**TODO**
- [ ] E1: compute and display the confusion matrix on the test set
- [ ] E2: find the 3 most-confused class pairs and show example misclassified images
- [ ] E3: explain *why* those confusions happen
- [ ] E4: report per-class accuracy; did the rare classes from A3 suffer?


In [ ]:
# E — gather predictions, then confusion matrix + per-class report
# Hint: loop the test loader collecting argmax predictions and true labels into two arrays.
#       from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix, classification_report

@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        ps.append(model(x.to(device)).argmax(1).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

y_true, y_pred = predict_all(final_model, test_loader)      # test_loader is unshuffled -> index i == test_ds[i]
print(f'Final test accuracy: {(y_true == y_pred).mean():.4f}')

def denorm(t):                                              # normalized tensor -> displayable HxWx3 array
    return (t * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

# ---- E1: confusion matrix ----
cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
plt.figure(figsize=(10, 9))
plt.imshow(np.log1p(cm), cmap='Blues')
plt.colorbar(label='log(1 + count)')
plt.xlabel('Predicted class'); plt.ylabel('True class'); plt.title('Confusion matrix (test set)')
plt.show()

# ---- E2: 3 most-confused pairs (largest off-diagonal entries) + example images ----
off = cm.copy(); np.fill_diagonal(off, 0)
top_pairs = [np.unravel_index(i, off.shape) for i in np.argsort(off.ravel())[::-1][:3]]
for t, p in top_pairs:
    print(f'\nTrue {t} ({CLASS_NAMES[t]})  ->  predicted {p} ({CLASS_NAMES[p]}): {off[t, p]} times')
    ex = np.where((y_true == t) & (y_pred == p))[0][:5]
    if len(ex):
        fig, axes = plt.subplots(1, len(ex), figsize=(2.2 * len(ex), 2.4))
        for ax, i in zip(np.atleast_1d(axes), ex):
            ax.imshow(denorm(test_ds[int(i)][0])); ax.axis('off')
        plt.suptitle(f'True: {CLASS_NAMES[t]}  |  Predicted: {CLASS_NAMES[p]}', fontsize=9)
        plt.show()

# ---- E4: per-class accuracy ----
per_class = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
colors = ['crimson' if c in RARE_CLASSES else 'steelblue' for c in range(NUM_CLASSES)]
plt.figure(figsize=(13, 4))
plt.bar(range(NUM_CLASSES), per_class, color=colors)
plt.ylim(0, 1.05); plt.xlabel('Class id'); plt.ylabel('Test accuracy'); plt.title('Per-class accuracy (red = 3 rarest training classes)')
plt.show()

print('5 hardest classes:')
for c in np.argsort(per_class)[:5]:
    print(f'  class {c:2d} ({CLASS_NAMES[c]}): {per_class[c]:.3f}  [{class_counts[c]} train images]')
print('\nRare classes from A3:')
for c in RARE_CLASSES:
    print(f'  class {c:2d} ({CLASS_NAMES[c]}): {per_class[c]:.3f}  [{class_counts[c]} train images]')

print('\n', classification_report(y_true, y_pred, labels=range(NUM_CLASSES),
      target_names=[f'{i}: {n}' for i, n in enumerate(CLASS_NAMES)], digits=3, zero_division=0))

**E3 + E4 analysis (write here):** _Why are those pairs confused? Did rare classes suffer?_

---
## Part F — Stretch goals  *(bonus +10%)*

**Hints**
- Robustness: wrap the test images with added Gaussian noise / blur / brightness shift and re-measure accuracy; plot accuracy vs. corruption strength.
- Explainability: Grad-CAM highlights the pixels that drove the prediction.
- Demo: softmax the logits for one image and print the top-3 classes with confidences.

**TODO (pick one or more)**
- [ ] Robustness curve  ·  [ ] Grad-CAM heatmap  ·  [ ] Top-3 single-image demo


In [ ]:
# F — your chosen stretch goal(s)   (all three implemented)
import torchvision.transforms.functional as TF

# ---------- 1) Robustness curves ----------
@torch.no_grad()
def accuracy_under(model, loader, corrupt):
    model.eval(); correct = n = 0
    for x, y in loader:
        x01 = corrupt((x * 0.5 + 0.5).to(device)).clamp(0, 1)      # corrupt in [0,1] pixel space
        x = (x01 - 0.5) / 0.5
        correct += (model(x).argmax(1).cpu() == y).sum().item(); n += y.size(0)
    return correct / n

noise_levels  = [0.0, 0.05, 0.1, 0.2, 0.3]
blur_levels   = [0.1, 0.5, 1.0, 1.5, 2.0]
bright_levels = [-0.4, -0.2, 0.0, 0.2, 0.4]
torch.manual_seed(SEED)
acc_noise  = [accuracy_under(final_model, test_loader, lambda x, s=s: x + s * torch.randn_like(x)) for s in noise_levels]
acc_blur   = [accuracy_under(final_model, test_loader, lambda x, s=s: TF.gaussian_blur(x, kernel_size=7, sigma=s)) for s in blur_levels]
acc_bright = [accuracy_under(final_model, test_loader, lambda x, d=d: x + d) for d in bright_levels]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, xs, ys, title, xl in [(axes[0], noise_levels, acc_noise, 'Gaussian noise', 'noise σ'),
                              (axes[1], blur_levels, acc_blur, 'Gaussian blur', 'blur σ'),
                              (axes[2], bright_levels, acc_bright, 'Brightness shift', 'shift')]:
    ax.plot(xs, ys, 'o-'); ax.set_title(title); ax.set_xlabel(xl); ax.set_ylabel('Test accuracy'); ax.set_ylim(0, 1.02); ax.grid(alpha=0.3)
plt.suptitle('Robustness of the final model'); plt.tight_layout(); plt.show()

# ---------- 2) Grad-CAM ----------
def grad_cam(model, x, target=None):
    """x: normalized tensor (3,H,W). Returns (heatmap in [0,1] at input size, class used)."""
    model.eval()
    store = {}
    layer = model.cam_layer()
    h1 = layer.register_forward_hook(lambda m, i, o: store.__setitem__('act', o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: store.__setitem__('grad', go[0]))
    logits = model(x.unsqueeze(0).to(device))
    c = int(logits.argmax(1)) if target is None else target
    model.zero_grad(); logits[0, c].backward()
    h1.remove(); h2.remove()
    w = store['grad'].mean(dim=(2, 3), keepdim=True)                     # channel importance
    cam = F.relu((w * store['act']).sum(1, keepdim=True))                # (1,1,h,w)
    cam = F.interpolate(cam, size=x.shape[1:], mode='bilinear', align_corners=False)[0, 0]
    cam = cam.detach().cpu().numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), c

pick = np.random.default_rng(1).choice(len(test_ds), 6, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for j, i in enumerate(pick):
    x, y = test_ds[int(i)]
    cam, c = grad_cam(final_model, x)
    axes[0, j].imshow(denorm(x)); axes[0, j].set_title(f'true: {y}', fontsize=9); axes[0, j].axis('off')
    axes[1, j].imshow(denorm(x)); axes[1, j].imshow(cam, cmap='jet', alpha=0.45)
    axes[1, j].set_title(f'pred: {c}', fontsize=9, color='green' if c == y else 'red'); axes[1, j].axis('off')
plt.suptitle('Grad-CAM: where the model looks (bottom row)'); plt.tight_layout(); plt.show()

# ---------- 3) Top-3 single-image demo ----------
final_model.eval()
i = int(pick[0]); x, y = test_ds[i]
with torch.no_grad():
    probs = F.softmax(final_model(x.unsqueeze(0).to(device)), dim=1)[0].cpu()
top_p, top_c = probs.topk(3)
plt.imshow(denorm(x)); plt.axis('off'); plt.title(f'True class: {y} ({CLASS_NAMES[y]})'); plt.show()
for p, c in zip(top_p, top_c):
    print(f'  class {int(c):2d} ({CLASS_NAMES[int(c)]}): {float(p):.1%}')

---
## Global hints
- Get the whole pipeline running on a **small subset** before scaling up — a model that trains badly beats a pipeline that won't run.
- "Better" only means something relative to your Part C baseline.
- Don't let test data leak into training or into augmentation statistics.
- `CrossEntropyLoss` wants raw logits; don't add a softmax before it.
- PyTorch: `model.train()` vs `model.eval()`, and `optimizer.zero_grad()` every step.


## What to submit & grading
**Submit** this completed notebook (code + outputs visible) and a short report (≈1–2 pages): preprocessing choices, the Part D comparison table, your confusion-matrix analysis, and a reflection on what helped most. State your **final test accuracy**.

| Component | Weight |
|---|---|
| A — Data exploration | 10% |
| B — Preprocessing | 10% |
| C — Baseline CNN | 25% |
| D — Improvements + comparison table | 30% |
| E — Evaluation & error analysis | 15% |
| Report clarity & reflection | 10% |
| F — Stretch goals | +10% bonus |

Grades reward **deliberate, measured** improvement and clear reasoning — not just the highest accuracy. A modest CNN reaches ~95%+ test accuracy here; well-tuned models exceed 98–99%.
